Embed the following query:

How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

-0.31
-0.02
0.12
0.44

In [2]:
from embedder import Embedder

embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(len(v))
print(v[0])

384
-0.02058203437252893


Answer Q1: -0.02

Loading the data
We pull the lesson pages from the course repository, the same way as in homework 1. We pin to commit 8c1834d so everyone works with the same data.

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
Each document is a dictionary with a filename and content, and there are 72 pages.

Q2. Cosine similarity
The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

0.07
0.37
0.68
0.92

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
from embedder import Embedder
from pathlib import Path
import numpy as np

embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
v_query = embedder.encode(query)

content = Path("07-sqlitesearch-vector.md").read_text(encoding="utf-8")

v_doc = embedder.encode(content)

similarity = np.dot(v_query, v_doc)

print(similarity)

0.36107027225589694


Answer Q2: 0.37

Q3. Chunking and search by hand
A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

scores = X.dot(v)
Which file does the highest-scoring chunk belong to (its filename)?

02-vector-search/lessons/03-embeddings-dataset.md
02-vector-search/lessons/06-rag-vector.md
02-vector-search/lessons/07-sqlitesearch-vector.md
02-vector-search/lessons/09-onnx-embedder.md

In [9]:
from gitsource import GithubRepositoryDataReader

help(GithubRepositoryDataReader)

Help on class GithubRepositoryDataReader in module gitsource.github:

class GithubRepositoryDataReader(builtins.object)
 |  GithubRepositoryDataReader(repo_owner: str, repo_name: str, commit_id: str | None = None, branch: str = 'main', allowed_extensions: Optional[Iterable[str]] = None, filename_filter: Optional[Callable[[str], bool]] = None, processors: dict[str, typing.Callable[[str, str], str]] | None = None, skip_hidden: bool = False) -> None
 |
 |  Downloads and parses files from a GitHub repository.
 |
 |  Uses codeload.github.com to fetch repository archives without requiring git.
 |
 |  Example:
 |      from gitsource import GithubRepositoryDataReader, notebook_processor
 |
 |      reader = GithubRepositoryDataReader(
 |          repo_owner="alexeygrigorev",
 |          repo_name="gitsource",
 |          allowed_extensions={"md", "ipynb"},
 |          processors={"ipynb": notebook_processor},
 |      )
 |      files = reader.read()
 |
 |  Methods defined here:
 |
 |  __init__(s

In [10]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    branch="main",
    allowed_extensions={"md"}
)

documents = list(reader.read())

print(len(documents))
print(documents[0].keys())

129


AttributeError: 'RawRepositoryFile' object has no attribute 'keys'

In [11]:
print(type(documents[0]))
print(dir(documents[0]))

<class 'gitsource.github.RawRepositoryFile'>
['__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'content', 'filename', 'parse']


In [12]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from embedder import Embedder
import numpy as np

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    branch="main",
    allowed_extensions={"md"},
)

documents_raw = list(reader.read())

documents = [
    {
        "filename": d.filename,
        "content": d.content,
    }
    for d in documents_raw
    if d.filename.startswith("02-vector-search/lessons/")
]

print("documents:", len(documents))
print(documents[0]["filename"])

chunks = chunk_documents(documents, size=2000, step=1000)

print("chunks:", len(chunks))

embedder = Embedder()

texts = [chunk["content"] for chunk in chunks]
X = embedder.encode_batch(texts)

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

scores = X.dot(v)
best_idx = scores.argmax()

print("best score:", scores[best_idx])
print("best file:", chunks[best_idx]["filename"])

documents: 10
02-vector-search/lessons/01-intro.md
chunks: 40
best score: 0.6489017718578813
best file: 02-vector-search/lessons/07-sqlitesearch-vector.md


Answer Q3: 02-vector-search/lessons/01-intro.md

Q4. Vector search with minsearch
We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

What metric do we use to evaluate a search engine?

Which file is the filename of the first result?

02-vector-search/lessons/04-vector-search.md
04-evaluation/lessons/05-search-metrics.md
04-evaluation/lessons/13-llm-as-judge.md
05-monitoring/lessons/04-metrics.md

In [13]:
from minsearch import VectorSearch

In [14]:
index = VectorSearch(
    keyword_fields=["filename"],
    text_fields=["content"],
    vector_field="embedding"
)

TypeError: VectorSearch.__init__() got an unexpected keyword argument 'text_fields'

In [15]:
from minsearch import VectorSearch
import inspect

print(inspect.signature(VectorSearch))

(keyword_fields=None, numeric_fields=None, date_fields=None)


In [16]:
from minsearch import VectorSearch

print(dir(VectorSearch))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'append', 'append_batch', 'fit', 'load', 'save', 'search']


In [17]:
from minsearch import VectorSearch
import inspect

print(inspect.signature(VectorSearch))

(keyword_fields=None, numeric_fields=None, date_fields=None)


In [18]:
from minsearch import VectorSearch

index = VectorSearch(keyword_fields=["filename"])

In [19]:
for chunk, embedding in zip(chunks, X):
    chunk["embedding"] = embedding

In [20]:
index.fit(chunks)

TypeError: VectorSearch.fit() missing 1 required positional argument: 'payload'

In [21]:
from minsearch import VectorSearch
import inspect

print("fit:", inspect.signature(VectorSearch.fit))
print("append:", inspect.signature(VectorSearch.append))
print("append_batch:", inspect.signature(VectorSearch.append_batch))
print("search:", inspect.signature(VectorSearch.search))

fit: (self, vectors, payload)
append: (self, vector, doc)
append_batch: (self, vectors, payload)
search: (self, query_vector, filter_dict=None, num_results=10, output_ids=False)


In [22]:
from minsearch import VectorSearch

index = VectorSearch(keyword_fields=["filename"])

index.fit(X, chunks)

In [23]:
query = "What metric do we use to evaluate a search engine?"
v_query = embedder.encode(query)

results = index.search(
    query_vector=v_query,
    num_results=5
)

print(results[0]["filename"])

02-vector-search/lessons/01-intro.md


In [24]:
print(len(chunks))
print(chunks[0]["filename"])
print(chunks[-1]["filename"])

40
02-vector-search/lessons/01-intro.md
02-vector-search/lessons/10-next-steps.md


In [25]:
print(results[0]["filename"])
print(results[0]["content"][:500])

02-vector-search/lessons/01-intro.md
ch uses an inverted index (BM25, TF-IDF). Vector search
  uses a vector index based on cosine similarity.
- Keyword search misses synonyms and paraphrases. Vector search misses
  exact term matches.

Vector search is usually better, but it adds a lot of operational
complexity, and you'll feel that throughout this module. So my advice
is to never start with vector search. Start with text search, and reach
for vectors once you can show they're worth the extra cost.

In practice the two work best t


Answer Q4: 02-vector-search/lessons/01-intro.md

Q5. Text search vs vector search
Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

02-vector-search/lessons/01-intro.md
02-vector-search/lessons/02-embeddings.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md

In [26]:
from minsearch import VectorSearch

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

query = "How do I store vectors in PostgreSQL?"
v_query = embedder.encode(query)

vector_results = vector_index.search(
    query_vector=v_query,
    num_results=5
)

vector_files = [r["filename"] for r in vector_results]
print(vector_files)

['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/02-embeddings.md']


In [27]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

text_results = text_index.search(
    query=query,
    num_results=5
)

text_files = [r["filename"] for r in text_results]
print(text_files)

['02-vector-search/lessons/01-intro.md', '02-vector-search/lessons/01-intro.md', '02-vector-search/lessons/02-embeddings.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/02-embeddings.md']


In [28]:
set(vector_files) - set(text_files)

set()

Answer Q5: 02-vector-search/lessons/02-embeddings.md

Q6. Hybrid search
Both vector and text search have their strengths and weaknesses. Vector search matches by meaning, so it finds relevant pages even when they use words different from the query. But it can miss exact terms like names, codes, or rare keywords. Text search is the opposite: it nails exact words but misses paraphrases and synonyms.

We don't have to pick one or the other - we can use both and merge their results. This approach is called "hybrid search".

Each search produces its own ranked list, so we need a way to combine them into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores the raw scores from each method, which live on different scales and aren't directly comparable. Instead, it looks only at the position of each document in each list.

Every document scores by its position (rank, starting at 0) in each list, and we sum the scores across lists with a constant k = 60:

RRF(d) = sum over lists of  1 / (k + rank(d))
"Sum over lists" means we go through every ranked list and, for each list where the document appears, add its 1 / (k + rank) contribution. A document found by both searches collects a score from each list, while one found by only a single search collects just one.

The constant k controls how much the exact rank matters. A larger k flattens the gap between positions, so the difference between rank 0 and rank 5 counts for less. A smaller k does the opposite: it sharpens that gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default. You rarely need to tune it. Lower it when only the top results matter. Raise it to reward documents that appear across many lists, even when they never quite reach the top.

A document that ranks well in both lists ends up higher than one that's only strong in a single list.

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]
Let's use this function.

Run text search and vector search (top 5 each) for this query, then combine them with rrf:

query = "How do I give the model access to tools?"
Which file is ranked first after RRF?

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/16-other-frameworks.md
Notice that this file isn't first in either search on its own - it wins because it ranks high in both.

In [29]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [30]:
query = "How do I give the model access to tools?"

v_query = embedder.encode(query)

vector_results = vector_index.search(
    query_vector=v_query,
    num_results=5
)

text_results = text_index.search(
    query=query,
    num_results=5
)

In [31]:
hybrid_results = rrf(
    [text_results, vector_results],
    k=60,
    num_results=5
)

for r in hybrid_results:
    print(r["filename"], r["start"])

02-vector-search/lessons/02-embeddings.md 3000
02-vector-search/lessons/08-pgvector.md 7000
02-vector-search/lessons/09-onnx-embedder.md 1000
02-vector-search/lessons/08-pgvector.md 6000
02-vector-search/lessons/06-rag-vector.md 1000


In [32]:
documents = [
    {
        "filename": d.filename,
        "content": d.content,
    }
    for d in documents_raw
    if d.filename.startswith("01-agentic-rag/lessons/")
]

In [33]:
chunks = chunk_documents(documents, size=2000, step=1000)

texts = [c["content"] for c in chunks]
X = embedder.encode_batch(texts)

In [34]:
vector_index.fit(X, chunks)

text_index.fit(chunks)

In [35]:
query = "How do I give the model access to tools?"

In [36]:
documents = [
    {
        "filename": d.filename,
        "content": d.content,
    }
    for d in documents_raw
    if d.filename.startswith("01-agentic-rag/lessons/")
]

print(len(documents))
print(documents[0]["filename"])

16
01-agentic-rag/lessons/01-intro.md


In [37]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))
print(chunks[0].keys())

76
dict_keys(['start', 'content', 'filename'])


In [38]:
texts = [c["content"] for c in chunks]
X = embedder.encode_batch(texts)

print(X.shape)

(76, 384)


In [39]:
from minsearch import VectorSearch

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

In [40]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [41]:
query = "How do I give the model access to tools?"

v_query = embedder.encode(query)

vector_results = vector_index.search(
    query_vector=v_query,
    num_results=5
)

text_results = text_index.search(
    query=query,
    num_results=5
)

In [42]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [43]:
hybrid_results = rrf(
    [text_results, vector_results],
    k=60,
    num_results=5
)

for r in hybrid_results:
    print(r["filename"], r["start"])

01-agentic-rag/lessons/01-intro.md 2000
01-agentic-rag/lessons/13-function-calling.md 4000
01-agentic-rag/lessons/14-agentic-loop.md 0
01-agentic-rag/lessons/16-other-frameworks.md 0
01-agentic-rag/lessons/13-function-calling.md 5000


Q6  answer : 01-agentic-rag/lessons/14-agentic-loop.md